# Randomized Smoothing — Certified Adversarial Robustness via Randomized Smoothing (2019)

_Papers · Meta-learning, Bayesian & Robustness_

**Add Gaussian noise, vote, and you get a classifier with a provable no-flip radius.**

---

This notebook is a practice scaffold for the **Randomized Smoothing — Certified Adversarial Robustness via Randomized Smoothing (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn
import numpy as np
from scipy.stats import norm, binomtest

torch.manual_seed(0); np.random.seed(0)

# --- 0. Worked example: plug p_A=0.99, p_B=0.01, sigma=0.5 into the radius formula. ---
sigma_we, pA_we = 0.5, 0.99
R_full = (sigma_we/2)*(norm.ppf(pA_we) - norm.ppf(1-pA_we))   # full Eqn. 3
R_two  = sigma_we*norm.ppf(pA_we)                              # two-class form
print("worked example: Phi^-1(0.99)=%.4f  R_full=%.4f  R_two=%.4f" % (
      norm.ppf(0.99), R_full, R_two))
# worked example: Phi^-1(0.99)=2.3263  R_full=1.1632  R_two=1.1632


# --- 1. A tiny 2D two-class dataset; base classifier = small MLP (composed with nn). ---
def make_data(n, seed=None):
    if seed is not None: np.random.seed(seed)
    y  = (np.random.rand(n) > 0.5).astype(int)
    cx = np.where(y == 1, 0.7, -0.7)                 # class set by sign of x[0]
    X  = np.stack([cx + np.random.randn(n)*0.55, np.random.randn(n)*0.55], 1)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y)

Xtr, ytr = make_data(500, 0)

def train(sigma_train, steps=500):
    torch.manual_seed(1)
    net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
    opt = torch.optim.Adam(net.parameters(), lr=0.05); lf = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        Xn = Xtr + torch.randn_like(Xtr)*sigma_train    # paper's prereq: train under noise
        lf(net(Xn), ytr).backward(); opt.step()
    return net

# --- 2. Randomized smoothing, BUILT BY HAND (the novel part). ---
def vote_counts(net, x, sigma, n):
    X = x.repeat(n, 1) + torch.randn(n, 2)*sigma         # n noisy copies, eps ~ N(0, sigma^2 I)
    with torch.no_grad():
        pred = net(X).argmax(1)
    c1 = int((pred == 1).sum()); return np.array([n - c1, c1])

def lower_conf(k, n, alpha=0.001):                       # binomial lower bound (Clopper-Pearson)
    return binomtest(k, n, p=0.5).proportion_ci(
        confidence_level=1 - 2*alpha, method="exact").low

def certify(net, x, sigma, n=8000, alpha=0.001):
    counts = vote_counts(net, x, sigma, n)
    cA = int(counts.argmax()); kA = int(counts[cA])
    pA = lower_conf(kA, n, alpha)                         # conservative top-class probability
    if pA <= 0.5:
        return cA, 0.0, pA                               # abstain: not a confident majority
    R = sigma*norm.ppf(pA)                               # two-class radius = sigma * Phi^-1(pA)
    return cA, R, pA

# --- 3. Effect A: radius grows with the MARGIN (distance from boundary), fixed sigma=0.5. ---
net5 = train(0.5)
print("\nRadius vs margin (sigma=0.5):")
for x0 in [0.2, 0.5, 1.0, 2.0]:
    cA, R, pA = certify(net5, torch.tensor([[x0, 0.0]]), 0.5)
    print("  x0=%.1f  pA=%.4f  R=%.4f" % (x0, pA, R))
# x0=0.2  pA=~0.64  R=~0.19
# x0=0.5  pA=~0.84  R=~0.49
# x0=1.0  pA=~0.97  R=~0.95
# x0=2.0  pA=~1.00  R=~1.57   <- farther from boundary -> larger certified radius

# --- 4. Effect B: larger sigma certifies LARGER radii (with an accuracy cost). ---
print("\nCertified accuracy at radius >= 0.5, per sigma (120 test points):")
Xte, yte = make_data(300, 7)
for s in [0.25, 0.5, 1.0]:
    ns = train(s); nrob = 0
    for i in range(120):
        cA, R, pA = certify(ns, Xte[i:i+1], s)
        if pA > 0.5 and cA == int(yte[i]) and R >= 0.5:
            nrob += 1
    print("  sigma=%.2f  cert_acc@R>=0.5 = %.3f" % (s, nrob/120))
# sigma=0.25 cert_acc@R>=0.5 = 0.000   <- small sigma CANNOT reach radius 0.5 here
# sigma=0.50 cert_acc@R>=0.5 = ~0.58
# sigma=1.00 cert_acc@R>=0.5 = ~0.53
# (Our small run, not the paper's reported numbers. Values vary by seed/hardware.)

## Visualize the data & results

_How does the certified-accuracy-versus-radius curve change as we raise the smoothing noise level sigma?_

In [ ]:
import torch, torch.nn as nn, numpy as np
from scipy.stats import norm, binomtest
torch.manual_seed(0); np.random.seed(0)

def make_data(n, seed=None):
    if seed is not None: np.random.seed(seed)
    y  = (np.random.rand(n) > 0.5).astype(int)
    cx = np.where(y == 1, 0.7, -0.7)
    X  = np.stack([cx + np.random.randn(n)*0.55, np.random.randn(n)*0.55], 1)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y)
Xtr, ytr = make_data(500, 0); Xte, yte = make_data(300, 7)

def train(st, steps=500):
    torch.manual_seed(1)
    net = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,2))
    opt = torch.optim.Adam(net.parameters(), lr=0.05); lf = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad(); Xn = Xtr + torch.randn_like(Xtr)*st
        lf(net(Xn), ytr).backward(); opt.step()
    return net

def cnt(net, x, s, n):
    X = x.repeat(n,1) + torch.randn(n,2)*s
    with torch.no_grad(): p = net(X).argmax(1)
    c1 = int((p==1).sum()); return np.array([n-c1, c1])
def lc(k, n, a=0.001):
    return binomtest(k, n, p=0.5).proportion_ci(confidence_level=1-2*a, method="exact").low
def cert(net, x, s, n=8000, a=0.001):
    c = cnt(net, x, s, n); cA = int(c.argmax()); kA = int(c[cA]); pA = lc(kA, n, a)
    if pA <= 0.5: return cA, 0.0, pA
    return cA, s*norm.ppf(pA), pA

grid = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
for s in [0.25, 0.5, 1.0]:
    ns = train(s); certR = []
    for i in range(120):
        cA, R, pA = cert(ns, Xte[i:i+1], s)
        certR.append(R if (pA > 0.5 and cA == int(yte[i])) else -1)
    certR = np.array(certR)
    row = [round(float((certR >= r).mean()), 3) for r in grid]
    print("sigma=%.2f:" % s, row)
# sigma=0.25: [0.908, 0.808, 0.617, 0.375, 0.0, 0.0]
# sigma=0.50: [0.908, 0.808, 0.617, 0.4, 0.258, 0.017]
# sigma=1.00: [0.908, 0.775, 0.608, 0.4, 0.258, 0.058]
# Larger sigma certifies larger radii; small sigma is capped. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Worked-number check. Under $\sigma = 1.0$, Monte-Carlo sampling at a point gives a top-class
            lower bound $\bar p_A = 0.95$ (two classes, so $\bar p_B = 0.05$). Compute the certified radius
            two ways and confirm they agree. Given $\Phi^{-1}(0.95) = 1.6449$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Full form: $R = \tfrac{\sigma}{2}(\Phi^{-1}(0.95) - \Phi^{-1}(0.05))$. By symmetry $\Phi^{-1}(0.05) = -1.6449$, so the gap is $1.6449 - (-1.6449) = 3.2898$. — _$\Phi^{-1}(1-p) = -\Phi^{-1}(p)$ because the standard normal is symmetric about $0$._
- Scale: $R = \tfrac{1.0}{2}\cdot 3.2898 = 0.5 \cdot 3.2898 = 1.6449$. — _The factor $\tfrac{\sigma}{2}$ converts the probability gap (in $\Phi^{-1}$ units) back to input distance._
- Two-class form: $R = \sigma\,\Phi^{-1}(\bar p_A) = 1.0 \cdot 1.6449 = 1.6449$. Same answer. — _With $\bar p_B = 1-\bar p_A$ the two halves of the gap are equal, so the half-gap times two cancels the $\tfrac12$._

**Answer:** Both give $R = 1.6449$. The certified radius equals $\sigma\,\Phi^{-1}(\bar p_A) =
                 1.0 \cdot 1.6449$. No $\ell_2$ perturbation shorter than $1.6449$ can flip this prediction.
                 Note it scales directly with $\sigma$: at $\sigma = 0.5$ the same vote would certify only
                 $0.822$.

</details>

**Problem 2.** The $\sigma$ trade-off (ablation). You certify the same test set at $\sigma = 0.25$ and at
            $\sigma = 1.0$. At small radii (near $0$) the low-$\sigma$ model has higher certified accuracy, but
            past some radius its certified accuracy drops to zero while the high-$\sigma$ model still
            certifies a fraction of points. Explain both halves.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Cap on the radius: with two classes $R = \sigma\,\Phi^{-1}(\bar p_A)$. Even a near-perfect vote ($\bar p_A$ close to $1$) gives a finite $\Phi^{-1}(\bar p_A)$, and the whole thing is multiplied by $\sigma$. — _A small $\sigma$ multiplies every radius down &mdash; so small-$\sigma$ certificates simply cannot reach large radii, no matter how confident the vote._
- Cost at small radii: raising $\sigma$ blurs the input with more noise, so the base classifier is less sure, $\bar p_A$ shrinks, and some easy points now abstain or certify a smaller radius. — _More noise erodes the vote margin, which lowers certified accuracy at the radii the low-$\sigma$ model already covered._
- Conclude: pick $\sigma$ for the radius you care about &mdash; small $\sigma$ for high accuracy at tiny radii, large $\sigma$ to certify larger perturbations. — _There is no single best $\sigma$; the certified-accuracy-versus-radius curves cross. This is the paper's central practical trade-off._

**Answer:** Two effects of $\sigma$ pull opposite ways. The radius is $\sigma\,\Phi^{-1}(\bar p_A)$, so
                 a small $\sigma$ caps how large a radius you can ever certify &mdash; past that cap its
                 certified accuracy hits zero. But a large $\sigma$ adds more noise, shrinking $\bar p_A$, so it
                 loses accuracy at the small radii the low-$\sigma$ model handled. The curves cross; you
                 choose $\sigma$ for the threat you care about. Our run below shows exactly this: $\sigma=0.25$
                 leads at $R=0.25$ but falls to $0$ by $R=1.0$, while $\sigma=1.0$ still certifies points out to
                 $R=1.5$.

</details>

**Problem 3.** Why does randomized smoothing need more Monte-Carlo samples $n$ to certify a larger
            radius, even at a point where the base classifier almost never errs under noise?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The radius uses $\bar p_A$, a lower confidence bound. With $n$ samples and confidence $1-\alpha$, the bound sits below the observed fraction by a gap that shrinks like $1/\sqrt{n}$. — _Fewer samples means a looser (more conservative) bound, so $\bar p_A$ is further below the true probability._
- To certify a large radius you need $\bar p_A$ very close to $1$ (since $\Phi^{-1}$ shoots up only as $p \to 1$). A loose bound caps how close to $1$ you can claim. — _$\Phi^{-1}(0.999)$ is far larger than $\Phi^{-1}(0.99)$; resolving that last fraction of a percent needs many samples._
- So even a perfect classifier needs huge $n$ to prove $\bar p_A \approx 0.9999$ and unlock a large radius. — _The certificate is limited by sampling confidence, not just by the classifier's true accuracy._

**Answer:** Because the certificate is only as strong as the lower bound $\bar p_A$ you can prove,
                 and that bound tightens like $1/\sqrt{n}$. Large radii require $\bar p_A$ pushed very close to
                 $1$ (where $\Phi^{-1}$ grows steeply), and resolving that requires many samples even for a
                 near-perfect base classifier. Big radii are expensive in samples, not just in $\sigma$.

</details>